# cat et stack : allonger un axe ou en créer un

Assembler deux objets sans confondre extraits et nouveaux groupes.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#concatener-ou-empiler).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous avons deux signaux d’un seul extrait, chacun de forme (1,3). Nous pouvons former un lot de deux extraits, ou conserver un groupe autour de chacun.

| Demande | Valeurs | Forme |
| --- | --- | --- |
| cat sur axe 0 | [[1,2,3],[4,5,6]] | (2,3) |
| stack sur axe 0 | [[[1,2,3]],[[4,5,6]]] | (2,1,3) |

cat allonge l’axe des extraits. stack ajoute un axe extérieur et conserve l’axe de taille un déjà présent.

## Démonstration 1 · Comparer les deux assemblages

Prévoir l’axe qui apparaîtra dans le résultat.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

a et b ont la même forme, avec des valeurs distinctes.

In [4]:
a = torch.tensor([[1., 2., 3.]])
b = torch.tensor([[4., 5., 6.]])


In [5]:
observer('a', a)
observer('b', b)

a forme (1, 3) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0, 3.0]]
b forme (1, 3) type torch.float32 requires_grad False
  valeurs : [[4.0, 5.0, 6.0]]


### Calculer l’état suivant

cat([a,b],dim=0) ajoute les lignes de b après celles de a.

In [6]:
nouveaux_extraits = torch.cat([a, b], dim=0)


In [7]:
observer('nouveaux_extraits', nouveaux_extraits)

nouveaux_extraits forme (2, 3) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]


### Calculer l’état suivant

stack crée un axe 0 de taille deux. Les axes d’origine suivent à ses positions 1 et 2.

In [8]:
nouveau_groupe = torch.stack([a, b], dim=0)


In [9]:
observer('nouveau_groupe', nouveau_groupe)

nouveau_groupe forme (2, 1, 3) type torch.float32 requires_grad False
  valeurs : [[[1.0, 2.0, 3.0]], [[4.0, 5.0, 6.0]]]


### Lire les sorties

Les deux assemblages créent un résultat sans modifier les valeurs de a et b.

In [10]:
print("cat", nouveaux_extraits.tolist(), tuple(nouveaux_extraits.shape))
print("stack", nouveau_groupe.tolist(), tuple(nouveau_groupe.shape))
print("source", a.tolist(), b.tolist())

cat [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]] (2, 3)
stack [[[1.0, 2.0, 3.0]], [[4.0, 5.0, 6.0]]] (2, 1, 3)
source [[1.0, 2.0, 3.0]] [[4.0, 5.0, 6.0]]


Le choix exprime le rôle voulu pour les axes. Ajouter des crochets n’est pas une opération purement esthétique.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Deux lots (3,2) sont concaténés sur dim=0. Quelle forme obtient-on ?

In [11]:
exo_a = torch.zeros(3,2)
exo_b = torch.ones(3,2)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.cat([exo_a,exo_b],dim=0).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

cat allonge un axe existant : 3+3 lignes.

</details>

### Exercice 2

Les mêmes lots sont empilés avec stack(...,dim=0). Quelle forme obtient-on ?

In [12]:
exo_a = torch.zeros(3,2)
exo_b = torch.ones(3,2)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.stack([exo_a,exo_b],dim=0).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

stack crée un nouvel axe qui compte les deux tableaux fournis.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_a = torch.zeros(3,2)
exo_b = torch.ones(3,2)
exo_forme = (6,2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.cat([exo_a,exo_b],dim=0).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_a = torch.zeros(3,2)
exo_b = torch.ones(3,2)
exo_forme = (2,3,2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.stack([exo_a,exo_b],dim=0).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [13]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()